##### ARTI 560 - Computer Vision

## Visual Representations with DINOv2 - Exercise

### Exercise 1: Unsupervised Clustering

In this exercise, you will use the `KMeans` algorithm from sklearn to group 20 images from the Oxford Pet dataset into 2 clusters (Cats vs. Dogs) based purely on their CLS tokens.

Instructions:

1.  Extract the 384-dimensional [CLS] tokens from 20 images of the Oxford-IIIT Pet dataset. Ensure your selection includes a mix of both cats and dogs.

2. Apply K-Means Clustering ($n=2$) to group the vectors based on mathematical similarity rather than provided labels.

3. Compare the predicted clusters against ground-truth labels.

In [ ]:
import os
from collections import Counter
from contextlib import nullcontext

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import torch
import torch.nn.functional as F
import sklearn.cluster._kmeans as sklearn_kmeans
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from torchvision.datasets import OxfordIIITPet
from transformers import AutoImageProcessor, AutoModel

# Local workaround for a macOS/OpenBLAS threadpoolctl issue in this environment.
sklearn_kmeans.threadpool_info = lambda: []
sklearn_kmeans.threadpool_limits = lambda *args, **kwargs: nullcontext()

MODEL_NAME = "facebook/dinov2-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
backbone.eval()

pet_data = OxfordIIITPet(
    root="data/oxford-iiit-pet",
    split="trainval",
    target_types="category",
    download=True,
)

CAT_BREEDS = {
    "Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair", "Egyptian Mau",
    "Maine Coon", "Persian", "Ragdoll", "Russian Blue", "Siamese", "Sphynx",
}

def pet_type(category_id):
    return "cat" if pet_data.classes[category_id] in CAT_BREEDS else "dog"

def collect_subset(dataset, per_label=10):
    cats, dogs = [], []
    seen = {"cat": set(), "dog": set()}
    for img, category in dataset:
        breed = dataset.classes[category]
        label = pet_type(category)
        bucket = cats if label == "cat" else dogs
        if len(bucket) < per_label and breed not in seen[label]:
            bucket.append((img.convert("RGB"), label, breed))
            seen[label].add(breed)
        if len(cats) == per_label and len(dogs) == per_label:
            break
    return [item for pair in zip(cats, dogs) for item in pair]

@torch.no_grad()
def encode_cls(image):
    encoded = processor(images=image, return_tensors="pt").to(DEVICE)
    tokens = backbone(**encoded).last_hidden_state
    vector = tokens[:, 0, :]
    return F.normalize(vector, p=2, dim=1).cpu().numpy().reshape(-1)

samples = collect_subset(pet_data, per_label=10)
X = np.asarray([encode_cls(image) for image, _, _ in samples])
labels = np.asarray([label for _, label, _ in samples])
y_true = np.where(labels == "cat", 0, 1)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

cluster_to_class = {}
for cluster in sorted(set(clusters)):
    values = y_true[clusters == cluster]
    cluster_to_class[cluster] = Counter(values).most_common(1)[0][0]

y_pred = np.asarray([cluster_to_class[cluster] for cluster in clusters])
pred_names = np.where(y_pred == 0, "cat", "dog")

print(f"CLS feature shape: {X.shape}")
print(f"Balanced sample counts: {dict(Counter(labels))}")
print(f"K-Means accuracy: {accuracy_score(y_true, y_pred):.2%}")
print("Confusion matrix rows=true [cat, dog], columns=predicted [cat, dog]:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

for i, (_, truth, breed) in enumerate(samples, start=1):
    print(f"{i:02d}. {breed:24s} true={truth:3s} cluster={clusters[i - 1]} predicted={pred_names[i - 1]}")


### Exercise 2: Image Classification with DINOv2

In this exercise you'll use a DINOv2 model with a pre-trained linear head to classify an image. You will observe how the model maps visual features to specific ImageNet-1k categories.

Instructions:
1. For this exercise, you must use the following Model ID. This specific checkpoint includes the necessary classification head trained on ImageNet-1k:

    Model ID: `facebook/dinov2-small-imagenet1k-1-layer`

2. Find an image online to make the inference. To ensure the model has a fair chance of success, the image should belong to one of the ImageNet-1k classes (e.g., a Golden Retriever, a grand piano, a school bus, or a coffee mug).

In [ ]:
import requests
import torch
from io import BytesIO
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "facebook/dinov2-small-imagenet1k-1-layer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
classifier = AutoModelForImageClassification.from_pretrained(MODEL_ID).to(DEVICE)
classifier.eval()

# a Labrador retriever.
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/640px-YellowLabradorLooking_new.jpg"
response = requests.get(url, timeout=20)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")

display(image.resize((360, 240)))

inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    logits = classifier(**inputs).logits[0]
    probs = torch.softmax(logits, dim=0)
    top_probs, top_ids = torch.topk(probs, 5)

print(f"Image URL: {url}")
print("Top-5 ImageNet predictions:")
for rank, (idx, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
    print(f"{rank}. {classifier.config.id2label[idx]}: {prob:.2%}")
